# **01. Khảo sát và Đánh giá Cấu trúc Dữ liệu (Data Exploration)**

Dự án Phân tích cảm xúc của nhãn hàng Vinamilk từ dữ liệu Shopee.

## **Mục tiêu:**
- Đọc dữ liệu thô từ file CSV đã được trích xuất.
- Kiểm tra phân bổ dữ liệu, các giá trị bị khuyết thiếu (NaN) và các dòng trùng lặp.
- Phân tách và dọn dẹp ban đầu các giá trị khuyết thiếu.

In [ ]:
import os
import sys
import pandas as pd

# Trỏ đến thư mục src để import module của dự án
sys.path.append(os.path.abspath('../src'))
import utils

### **1. Nạp dữ liệu thô**
Đọc file dữ liệu Shopee đã được xuất từ Excel sang CSV.

In [ ]:
raw_reviews_path = "../data/raw/shopee_reviews.csv"

if not os.path.exists(raw_reviews_path):
    print(f"❌ Không tìm thấy file: {raw_reviews_path}")
else:
    df = pd.read_csv(raw_reviews_path, encoding="utf-8-sig")
    print(f"✅ Tải dữ liệu thành công: {len(df):,} dòng")
    display(df.head(3))

### **2. Kiểm tra Cấu trúc Dữ liệu (Schema) & Dữ liệu Thiếu**

In [ ]:
print("--- Cấu trúc dữ liệu ---")
df.info()

print("\n--- Số lượng giá trị Null ở từng cột ---")
print(df.isnull().sum())

### **3. Xử lý các giá trị khuyết thiếu**
- Điền tên người dùng bị trống bằng `'unknown user'`.
- Loại bỏ hoàn toàn các dòng bình luận bị trống chữ (`comment` bị Null).

In [ ]:
# Điền tên khuyết
null_user_count = df["user"].isnull().sum()
df["user"] = df["user"].fillna("unknown user")
print(f"-> Đã xử lý {null_user_count} dòng khuyết tên người dùng.")

# Loại bỏ comment khuyết
null_comment_count = df["comment"].isnull().sum()
df = df.dropna(subset=["comment"]).reset_index(drop=True)
print(f"-> Đã loại bỏ {null_comment_count} dòng bình luận trống chữ.")

### **4. Phân tách bình luận chỉ chấm sao không viết chữ**
Tách riêng các đánh giá mặc định (`"(Chỉ đánh giá sao, không viết nội dung)"`) ra khỏi bình luận chữ để lưu trữ riêng biệt.

In [ ]:
star_only_mask = df["comment"].str.strip() == "(Chỉ đánh giá sao, không viết nội dung)"
star_only_reviews = df[star_only_mask].copy()
text_reviews = df[~star_only_mask].copy()

print(f"📊 Thống kê bình luận sau phân tách:")
print(f"   + Bình luận chỉ chấm sao: {len(star_only_reviews):,} dòng")
print(f"   + Bình luận có nội dung chữ: {len(text_reviews):,} dòng")

### **5. Lưu kết quả sơ bộ**
Lưu tập bình luận chữ vào thư mục `data/processed/` để thực hiện làm sạch ở giai đoạn sau.

In [ ]:
text_reviews.to_csv("../data/processed/text_reviews.csv", index=False, encoding="utf-8-sig")
print("✅ Đã lưu tập bình luận chữ vào ../data/processed/text_reviews.csv")